# Reposition self

Example of how to control users relative position in the scene without e.g moving the simulation itself. This is useful for recordings that contains avatars interacting with the box in a static location.

<video controls src="../figures/reposition-user.webm">

## Setup runner & utilities

In [1]:
from nanover.app import OmniRunner
from nanover.recording import PlaybackSimulation

simulation = PlaybackSimulation.from_path(path="../systems/recordings/nanotube-methane-01.nanover.zip")
simulation.load()

OmniRunner.close_all_runners()
imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: user positioning")
imd_runner.load(0)

In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

## Reposition mode

In [10]:
import math
import numpy as np

from MDAnalysis.lib.transformations import rotation_matrix, quaternion_matrix, quaternion_from_matrix
from nanover.jupyter import Mode

def get_first_avatar_position_rotation():
    avatar = next((value for key, value in utilities.copy_shared_state().items() if key.startswith("avatar.")))
    head = next((value for value in avatar["components"] if "headset" in value["name"]))

    return head["position"], head["rotation"]

def snap_quaternion_to_y_axis(quaternion):
    snapped = np.array([0, quaternion[1], 0, quaternion[3]])
    snapped /= np.linalg.norm(snapped)
    return snapped

class FreeMoveMode(Mode):
    def on_cursor_updated(self, *, key: str, cursor: dict):
        # get look for joystick and which user and hand it belongs to
        _, user_id, hand = key.split(".")
        x, y = cursor.get("joystick", [0, 0])

        # find that users current position
        origin_key = f"user-origin.{user_id}"
        origin = utilities.get_shared_state_value(origin_key, dict(position=[0, 0, 0], rotation=[0, 0, 0, 1]))

        if hand == "left":
            # find the planar rotation of this user's headset
            avatar = utilities.get_shared_state_value(f"avatar.{user_id}")
            head = next((value for value in avatar["components"] if "headset" in value["name"]))
            q = snap_quaternion_to_y_axis(head["rotation"])
            rot = quaternion_matrix([q[3], *q[:3]])

            # add left joystick x/y motion in the facing direction
            move = rot @ [-x, 0, -y, 1]
            origin["position"] = move[:3] * 1/30 + origin["position"]
        elif hand == "right":
            # extract their current rotation matrix
            q = origin["rotation"]
            rot = quaternion_matrix([q[3], *q[:3]])

            # add right joystick x motion to rotation on the y-axis
            turn = rotation_matrix(math.pi * 1/30 * x, (0, -1, 0))
            q = quaternion_from_matrix(turn @ rot)
            origin["rotation"] = [*q[1:], q[0]]

        # apply updated position and rotation
        utilities.set_shared_state_value(origin_key, origin)

utilities.add_interaction_mode(FreeMoveMode(), "free move", icon="✳️")
utilities.modes.enter_mode("free move")